# Sprawozdanie z projektu: Sztuczne Sieci Neuronowe i Uczenie Maszynowe
Zbiór danych „Wine Quality Dataset” zawiera ponad 6400 obserwacji atrybutów fizykochemicznych wina. Projekt dzieli się na dwa problemy badawcze:
1. **Problem klasyfikacyjny (sieć zwraca klasę):** Przewidywanie kategorii jakości wina (słabe, średnie, dobre).
2. **Problem regresyjny (sieć zwraca wartość liczbową):** Przewidywanie dokładnej zawartości alkoholu.

## Przegląd literatury
Tematyka przewidywania jakości wina na podstawie testów fizykochemicznych była szeroko badana w literaturze. Fundamentalnym opracowaniem jest praca *Cortez et al. (2009), "Modeling wine preferences by data mining from physicochemical properties"* (Decis. Support Syst., 47(4):547-553), w której udowodniono skuteczność metod takich jak Random Forest w modelowaniu preferencji smakowych bez angażowania sommelierów. W niniejszym projekcie rozszerzamy tę tematykę o implementację Sztucznej Sieci Neuronowej (MLP) stworzonej od podstaw.

In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import mean_squared_error, accuracy_score
import warnings
warnings.filterwarnings('ignore') # Ukrywa ostrzeżenia z bibliotek

# Wczytanie danych
df = pd.read_csv('WineQT.csv')
if 'Id' in df.columns:
    df = df.drop('Id', axis=1)

# Przygotowanie celów (Klasyfikacja: 0-słabe, 1-średnie, 2-dobre)
df['quality_class'] = pd.cut(df['quality'], bins=[0, 4, 6, 10], labels=[0, 1, 2], include_lowest=True)

# Rozdzielenie danych na Cechy i Cele
X = df.drop(['quality', 'quality_class'], axis=1)
y_class = df['quality_class'].values
y_reg = df['alcohol'].values
X_reg = X.drop('alcohol', axis=1) # Do zgadywania alkoholu usuwamy go z cech

# Automatyczny podział danych (80% Train, 20% Test) i Normalizacja (StandardScaler)
scaler_r = StandardScaler()
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
X_train_r = scaler_r.fit_transform(X_train_r)
X_test_r = scaler_r.transform(X_test_r)

scaler_c = StandardScaler()
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_class, test_size=0.2, random_state=42)
X_train_c = scaler_c.fit_transform(X_train_c)
X_test_c = scaler_c.transform(X_test_c)

print(f"Dane gotowe. Uczymy się na {X_train_r.shape[0]} wierszach, testujemy na {X_test_r.shape[0]} wierszach.")

Dane gotowe. Uczymy się na 914 wierszach, testujemy na 229 wierszach.


## Część 1: Sztuczne Sieci Neuronowe (SSN)
Sieci neuronowe zostały zaimplementowane samodzielnie w języku Python, bez wykorzystania gotowych bibliotek do uczenia modeli (korzystano jedynie z biblioteki `numpy` do operacji macierzowych). Zespół projektowy składa się z 3 osób, w związku z czym dla każdego problemu przetestowano minimum 6 różnych parametrów. W przypadku każdego parametru sprawdzono 4 różne wartości. Uczenie powtórzono 3-krotnie i uśredniono.

Poniżej zaimplementowano dwa oddzielne silniki: dla klasyfikacji (na wyjściu aktywacja Softmax) oraz dla regresji.

In [27]:
# SILNIK SIECI DLA REGRESJI
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01):
        self.W1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.1
        self.b2 = np.zeros((1, output_size))
        self.lr = learning_rate

    def sigmoid(self, z): return 1 / (1 + np.exp(-np.clip(z, -250, 250)))
    def sigmoid_derivative(self, z): return z * (1 - z)

    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.output = self.z2
        return self.output

    def backward(self, X, y, output):
        m = X.shape[0]
        d_output = output - y.reshape(-1, 1)
        dW2 = (1 / m) * np.dot(self.a1.T, d_output)
        db2 = (1 / m) * np.sum(d_output, axis=0, keepdims=True)
        d_a1 = np.dot(d_output, self.W2.T)
        d_z1 = d_a1 * self.sigmoid_derivative(self.a1)
        dW1 = (1 / m) * np.dot(X.T, d_z1)
        db1 = (1 / m) * np.sum(d_z1, axis=0, keepdims=True)
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            self.backward(X, y, self.forward(X))


# SILNIK SIECI DLA KLASYFIKACJI
class NeuralNetworkClassifier(NeuralNetwork):
    def softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.output = self.softmax(self.z2)
        return self.output

    def backward(self, X, y, output):
        m = X.shape[0]
        d_output = output - y
        dW2 = (1 / m) * np.dot(self.a1.T, d_output)
        db2 = (1 / m) * np.sum(d_output, axis=0, keepdims=True)
        d_a1 = np.dot(d_output, self.W2.T)
        d_z1 = d_a1 * self.sigmoid_derivative(self.a1)
        dW1 = (1 / m) * np.dot(X.T, d_z1)
        db1 = (1 / m) * np.sum(d_z1, axis=0, keepdims=True)
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

def to_one_hot(y, num_classes=3):
    return np.eye(num_classes)[y.astype(int)]

In [24]:
print("--- TESTY SSN: PROBLEM 1 (REGRESJA - Błąd MSE) ---")
input_size = X_train_r.shape[1]
output_size = 1
powtorzenia = 3
X_reg_np = X_reg.values # Wersja bez normalizacji

def calc_mse(y_true, y_pred): return np.mean((y_true.reshape(-1, 1) - y_pred) ** 2)

print("\nParametr 1: Liczba neuronów (LR=0.01, Epoki=500)")
for hidden in [8, 16, 32, 64]:
    e_tr, e_te = [], []
    for _ in range(powtorzenia):
        nn = NeuralNetwork(input_size, hidden, output_size, 0.01)
        nn.train(X_train_r, y_train_r, 500)
        e_tr.append(calc_mse(y_train_r, nn.forward(X_train_r)))
        e_te.append(calc_mse(y_test_r, nn.forward(X_test_r)))
    print(f"Neurony {hidden:2d} | MSE Train: {np.mean(e_tr):.4f} | MSE Test: {np.mean(e_te):.4f}")

print("\nParametr 2: Współczynnik uczenia (Neurony=16, Epoki=500)")
for lr in [0.1, 0.05, 0.01, 0.001]:
    e_tr, e_te = [], []
    for _ in range(powtorzenia):
        nn = NeuralNetwork(input_size, 16, output_size, lr)
        nn.train(X_train_r, y_train_r, 500)
        e_tr.append(calc_mse(y_train_r, nn.forward(X_train_r)))
        e_te.append(calc_mse(y_test_r, nn.forward(X_test_r)))
    print(f"LR {lr:.3f} | MSE Train: {np.mean(e_tr):.4f} | MSE Test: {np.mean(e_te):.4f}")

print("\nParametr 3: Liczba epok (Neurony=16, LR=0.01)")
for eps in [100, 500, 1000, 2000]:
    e_tr, e_te = [], []
    for _ in range(powtorzenia):
        nn = NeuralNetwork(input_size, 16, output_size, 0.01)
        nn.train(X_train_r, y_train_r, eps)
        e_tr.append(calc_mse(y_train_r, nn.forward(X_train_r)))
        e_te.append(calc_mse(y_test_r, nn.forward(X_test_r)))
    print(f"Epoki {eps:4d} | MSE Train: {np.mean(e_tr):.4f} | MSE Test: {np.mean(e_te):.4f}")

print("\nParametr 4: Proporcja podziału zbioru (Train/Test)")
for split_ratio in [0.5, 0.6, 0.7, 0.8]:
    err_tr, err_te = [], []
    test_size_local = int(len(X_reg_np) * (1 - split_ratio))
    X_tr, X_te = X_reg_np[test_size_local:], X_reg_np[:test_size_local]
    y_tr, y_te = y_reg[test_size_local:], y_reg[:test_size_local]

    for _ in range(powtorzenia):
        nn = NeuralNetwork(input_size, 16, output_size, 0.01)
        nn.train(X_tr, y_tr, 500)
        err_tr.append(calc_mse(y_tr, nn.forward(X_tr)))
        err_te.append(calc_mse(y_te, nn.forward(X_te)))
    print(f"Train {int(split_ratio*100)}% / Test {int((1-split_ratio)*100)}% | MSE Train: {np.mean(err_tr):.4f} | MSE Test: {np.mean(err_te):.4f}")

print("\nParametr 5: Sposób doboru próby (Losowy vs Sekwencyjny)")
metody = ["Losowy (Złoty Standard)", "Sekwencyjny (Brak przemieszania)"]
for metoda in metody:
    err_tr, err_te = [], []
    idx = np.random.permutation(len(X_reg_np)) if "Losowy" in metoda else np.arange(len(X_reg_np))
    t_size = int(len(X_reg_np) * 0.2)
    X_tr, X_te = X_reg_np[idx[t_size:]], X_reg_np[idx[:t_size]]
    y_tr, y_te = y_reg[idx[t_size:]], y_reg[idx[:t_size]]

    for _ in range(powtorzenia):
        nn = NeuralNetwork(input_size, 16, output_size, 0.01)
        nn.train(X_tr, y_tr, 500)
        err_tr.append(calc_mse(y_tr, nn.forward(X_tr)))
        err_te.append(calc_mse(y_te, nn.forward(X_te)))
    print(f"Dobór: {metoda[:10]:10s} | MSE Train: {np.mean(err_tr):.4f} | MSE Test: {np.mean(err_te):.4f}")

print("\nParametr 6: Wpływ normalizacji danych wejściowych")
dane_wersje = {"Znormalizowane (0-1)": X_train_r, "Brak normalizacji": X_reg_np}
for nazwa, X_data in dane_wersje.items():
    err_tr, err_te = [], []
    idx = np.random.permutation(len(X_data))
    t_size = int(len(X_data) * 0.2)
    X_tr, X_te = X_data[idx[t_size:]], X_data[idx[:t_size]]
    y_tr, y_te = y_reg[idx[t_size:]], y_reg[idx[:t_size]]

    for _ in range(powtorzenia):
        nn = NeuralNetwork(input_size, 16, output_size, 0.01)
        nn.train(X_tr, y_tr, 500)
        err_tr.append(calc_mse(y_tr, nn.forward(X_tr)))
        err_te.append(calc_mse(y_te, nn.forward(X_te)))
    print(f"Dane: {nazwa:22s} | MSE Train: {np.mean(err_tr):.4f} | MSE Test: {np.mean(err_te):.4f}")

--- TESTY SSN: PROBLEM 1 (REGRESJA - Błąd MSE) ---

Parametr 1: Liczba neuronów (LR=0.01, Epoki=500)
Neurony  8 | MSE Train: 0.4219 | MSE Test: 0.5379
Neurony 16 | MSE Train: 0.4381 | MSE Test: 0.5586
Neurony 32 | MSE Train: 0.4703 | MSE Test: 0.6021
Neurony 64 | MSE Train: 0.5410 | MSE Test: 0.6905

Parametr 2: Współczynnik uczenia (Neurony=16, Epoki=500)
LR 0.100 | MSE Train: 0.3199 | MSE Test: 0.4156
LR 0.050 | MSE Train: 0.3393 | MSE Test: 0.4392
LR 0.010 | MSE Train: 0.4383 | MSE Test: 0.5566
LR 0.001 | MSE Train: 1.3015 | MSE Test: 1.4710

Parametr 3: Liczba epok (Neurony=16, LR=0.01)
Epoki  100 | MSE Train: 0.7746 | MSE Test: 0.9224
Epoki  500 | MSE Train: 0.4310 | MSE Test: 0.5514
Epoki 1000 | MSE Train: 0.3701 | MSE Test: 0.4743
Epoki 2000 | MSE Train: 0.3404 | MSE Test: 0.4408

Parametr 4: Proporcja podziału zbioru (Train/Test)
Train 50% / Test 50% | MSE Train: 1.1022 | MSE Test: 1.3947
Train 60% / Test 40% | MSE Train: 1.1245 | MSE Test: 1.2189
Train 70% / Test 30% | MSE Tra

In [25]:
print("--- TESTY SSN: PROBLEM 2 (KLASYFIKACJA - Skuteczność Accuracy) ---")
in_c = X_train_c.shape[1]
out_c = 3
y_tr_oh = to_one_hot(y_train_c)
y_te_oh = to_one_hot(y_test_c)
X_class_np = X.values # Do testów proporcji podziału

def calc_acc(y_true, y_pred): return np.mean(y_true.astype(int) == y_pred) * 100

print("\nParametr 1: Liczba neuronów (LR=0.1, Epoki=500)")
for hidden in [8, 16, 32, 64]:
    a_tr, a_te = [], []
    for _ in range(powtorzenia):
        nnc = NeuralNetworkClassifier(in_c, hidden, out_c, 0.1)
        nnc.train(X_train_c, y_tr_oh, 500)
        a_tr.append(calc_acc(y_train_c, nnc.predict(X_train_c)))
        a_te.append(calc_acc(y_test_c, nnc.predict(X_test_c)))
    print(f"Neurony {hidden:2d} | Acc Train: {np.mean(a_tr):.2f}% | Acc Test: {np.mean(a_te):.2f}%")

print("\nParametr 2: Współczynnik uczenia (Neurony=16, Epoki=500)")
for lr in [0.5, 0.1, 0.05, 0.01]:
    a_tr, a_te = [], []
    for _ in range(powtorzenia):
        nnc = NeuralNetworkClassifier(in_c, 16, out_c, lr)
        nnc.train(X_train_c, y_tr_oh, 500)
        a_tr.append(calc_acc(y_train_c, nnc.predict(X_train_c)))
        a_te.append(calc_acc(y_test_c, nnc.predict(X_test_c)))
    print(f"LR {lr:.3f} | Acc Train: {np.mean(a_tr):.2f}% | Acc Test: {np.mean(a_te):.2f}%")

print("\nParametr 3: Liczba epok (Neurony=16, LR=0.1)")
for eps in [100, 500, 1000, 2000]:
    a_tr, a_te = [], []
    for _ in range(powtorzenia):
        nnc = NeuralNetworkClassifier(in_c, 16, out_c, 0.1)
        nnc.train(X_train_c, y_tr_oh, eps)
        a_tr.append(calc_acc(y_train_c, nnc.predict(X_train_c)))
        a_te.append(calc_acc(y_test_c, nnc.predict(X_test_c)))
    print(f"Epoki {eps:4d} | Acc Train: {np.mean(a_tr):.2f}% | Acc Test: {np.mean(a_te):.2f}%")

print("\nParametr 4: Proporcja podziału zbioru (Train/Test)")
for split_ratio in [0.5, 0.6, 0.7, 0.8]:
    acc_tr, acc_te = [], []
    test_size_local = int(len(X_class_np) * (1 - split_ratio))
    X_tr, X_te = X_class_np[test_size_local:], X_class_np[:test_size_local]
    y_tr, y_te = y_class[test_size_local:], y_class[:test_size_local]
    y_tr_oh_loc = to_one_hot(y_tr)

    for _ in range(powtorzenia):
        nn_c = NeuralNetworkClassifier(in_c, 16, out_c, 0.1)
        nn_c.train(X_tr, y_tr_oh_loc, 500)
        acc_tr.append(calc_acc(y_tr, nn_c.predict(X_tr)))
        acc_te.append(calc_acc(y_te, nn_c.predict(X_te)))
    print(f"Train {int(split_ratio*100)}% / Test {int((1-split_ratio)*100)}% | Acc Train: {np.mean(acc_tr):.2f}% | Acc Test: {np.mean(acc_te):.2f}%")

print("\nParametr 5: Sposób doboru próby (Losowy vs Sekwencyjny)")
for metoda in ["Losowy", "Sekwencyjny"]:
    acc_tr, acc_te = [], []
    idx = np.random.permutation(len(X_class_np)) if metoda == "Losowy" else np.arange(len(X_class_np))
    t_size = int(len(X_class_np) * 0.2)
    X_tr, X_te = X_class_np[idx[t_size:]], X_class_np[idx[:t_size]]
    y_tr, y_te = y_class[idx[t_size:]], y_class[idx[:t_size]]
    y_tr_oh_loc = to_one_hot(y_tr)

    for _ in range(powtorzenia):
        nn_c = NeuralNetworkClassifier(in_c, 16, out_c, 0.1)
        nn_c.train(X_tr, y_tr_oh_loc, 500)
        acc_tr.append(calc_acc(y_tr, nn_c.predict(X_tr)))
        acc_te.append(calc_acc(y_te, nn_c.predict(X_te)))
    print(f"Dobór: {metoda:10s} | Acc Train: {np.mean(acc_tr):.2f}% | Acc Test: {np.mean(acc_te):.2f}%")

print("\nParametr 6: Metoda uczenia (Wpływ ilości epok przy małym LR=0.01)")
for eps in [500, 1000, 2000, 4000]:
    acc_tr, acc_te = [], []
    for _ in range(powtorzenia):
        nn_c = NeuralNetworkClassifier(in_c, 32, out_c, 0.01)
        nn_c.train(X_train_c, y_tr_oh, eps)
        acc_tr.append(calc_acc(y_train_c, nn_c.predict(X_train_c)))
        acc_te.append(calc_acc(y_test_c, nnc.predict(X_test_c)))
    print(f"Epoki: {eps:4d} (przy LR=0.01) | Acc Train: {np.mean(acc_tr):.2f}% | Acc Test: {np.mean(acc_te):.2f}%")

--- TESTY SSN: PROBLEM 2 (KLASYFIKACJA - Skuteczność Accuracy) ---

Parametr 1: Liczba neuronów (LR=0.1, Epoki=500)
Neurony  8 | Acc Train: 82.75% | Acc Test: 85.59%
Neurony 16 | Acc Train: 84.10% | Acc Test: 85.88%
Neurony 32 | Acc Train: 84.14% | Acc Test: 87.05%
Neurony 64 | Acc Train: 84.03% | Acc Test: 87.63%

Parametr 2: Współczynnik uczenia (Neurony=16, Epoki=500)
LR 0.500 | Acc Train: 84.17% | Acc Test: 89.67%
LR 0.100 | Acc Train: 84.06% | Acc Test: 86.03%
LR 0.050 | Acc Train: 82.06% | Acc Test: 85.01%
LR 0.010 | Acc Train: 82.06% | Acc Test: 85.15%

Parametr 3: Liczba epok (Neurony=16, LR=0.1)
Epoki  100 | Acc Train: 82.06% | Acc Test: 85.15%
Epoki  500 | Acc Train: 84.14% | Acc Test: 86.32%
Epoki 1000 | Acc Train: 83.99% | Acc Test: 88.50%
Epoki 2000 | Acc Train: 84.10% | Acc Test: 89.08%

Parametr 4: Proporcja podziału zbioru (Train/Test)
Train 50% / Test 50% | Acc Train: 79.55% | Acc Test: 85.81%
Train 60% / Test 40% | Acc Train: 82.22% | Acc Test: 83.37%
Train 70% / Test

## Część 2: Uczenie Maszynowe (Gotowe Biblioteki)
Wykorzystano algorytmy z biblioteki Scikit-learn, testując 3 parametry dla każdej metody.

**1. Las Losowy (ang. Random Forest)**
Metoda Lasu Losowego to potężny algorytm uczenia zespołowego (ensemble learning), stosowany zarówno do zadań klasyfikacji, jak i regresji. Zamiast opierać się na jednym modelu, algorytm ten tworzy "las" składający się z wielu pojedynczych drzew decyzyjnych. Każde drzewo jest trenowane na losowo wybranym podzbiorze danych uczących. Zaletą tej metody jest wysoka skuteczność, stabilność oraz duża odporność na problem przeuczenia (overfitting).

**2. K-Najbliższych Sąsiadów (ang. K-Nearest Neighbors - KNN)**
Algorytm KNN to skuteczna metoda tzw. uczenia leniwego (lazy learning). W przeciwieństwie do innych algorytmów, KNN nie buduje wewnętrznego modelu matematycznego w fazie treningu, lecz "zapamiętuje" cały zbiór uczący. Decyzja o przypisaniu nowej obserwacji opiera się na obliczeniu odległości (np. euklidesowej) między nowym punktem a wszystkimi punktami w bazie i sprawdzeniu wyników dla *K* najbliższych sąsiadów.

In [28]:
print("=======================================================")
print("  CZĘŚĆ 2: PROBLEM 1 (REGRESJA - Wartość Alkoholu) ")
print("=======================================================\n")

print("METODA A: Las Losowy (Random Forest Regressor)")
print("Parametr 1: Liczba drzew (n_estimators)")
for n in [10, 50, 100, 200]:
    rf = RandomForestRegressor(n_estimators=n, random_state=42)
    rf.fit(X_train_r, y_train_r)
    print(f"Drzewa: {n:3d} | MSE Train: {mean_squared_error(y_train_r, rf.predict(X_train_r)):.4f} | MSE Test: {mean_squared_error(y_test_r, rf.predict(X_test_r)):.4f}")

print("\nParametr 2: Maksymalna głębokość drzewa (max_depth)")
for d in [5, 10, 20, None]:  # None oznacza brak limitu
    rf = RandomForestRegressor(max_depth=d, random_state=42)
    rf.fit(X_train_r, y_train_r)
    d_str = str(d) if d is not None else "Brak"
    print(f"Głębokość: {d_str:>4s} | MSE Train: {mean_squared_error(y_train_r, rf.predict(X_train_r)):.4f} | MSE Test: {mean_squared_error(y_test_r, rf.predict(X_test_r)):.4f}")

print("\nParametr 3: Minimalna liczba próbek w liściu (min_samples_leaf)")
for m in [1, 2, 5, 10]:
    rf = RandomForestRegressor(min_samples_leaf=m, random_state=42)
    rf.fit(X_train_r, y_train_r)
    print(f"Próbki w liściu: {m:2d} | MSE Train: {mean_squared_error(y_train_r, rf.predict(X_train_r)):.4f} | MSE Test: {mean_squared_error(y_test_r, rf.predict(X_test_r)):.4f}")


print("\n\nMETODA B: K-Najbliższych Sąsiadów (KNN Regressor)")
print("Parametr 1: Liczba sąsiadów (n_neighbors)")
for k in [3, 5, 9, 15]:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_r, y_train_r)
    print(f"Sąsiedzi: {k:2d} | MSE Train: {mean_squared_error(y_train_r, knn.predict(X_train_r)):.4f} | MSE Test: {mean_squared_error(y_test_r, knn.predict(X_test_r)):.4f}")

print("\nParametr 2: Metryka odległości (p)")
# p=1: Manhattan, p=2: Euklidesowa
for p_val in [1, 2, 3, 4]:
    knn = KNeighborsRegressor(p=p_val)
    knn.fit(X_train_r, y_train_r)
    print(f"Metryka p: {p_val:2d} | MSE Train: {mean_squared_error(y_train_r, knn.predict(X_train_r)):.4f} | MSE Test: {mean_squared_error(y_test_r, knn.predict(X_test_r)):.4f}")

print("\nParametr 3: Algorytm wyszukiwania sąsiadów")
for alg in ['ball_tree', 'kd_tree', 'brute', 'auto']:
    knn = KNeighborsRegressor(algorithm=alg)
    knn.fit(X_train_r, y_train_r)
    print(f"Algorytm: {alg:9s} | MSE Train: {mean_squared_error(y_train_r, knn.predict(X_train_r)):.4f} | MSE Test: {mean_squared_error(y_test_r, knn.predict(X_test_r)):.4f}")

  CZĘŚĆ 2: PROBLEM 1 (REGRESJA - Wartość Alkoholu) 

METODA A: Las Losowy (Random Forest Regressor)
Parametr 1: Liczba drzew (n_estimators)
Drzewa:  10 | MSE Train: 0.0554 | MSE Test: 0.4270
Drzewa:  50 | MSE Train: 0.0403 | MSE Test: 0.3839
Drzewa: 100 | MSE Train: 0.0384 | MSE Test: 0.3780
Drzewa: 200 | MSE Train: 0.0377 | MSE Test: 0.3784

Parametr 2: Maksymalna głębokość drzewa (max_depth)
Głębokość:    5 | MSE Train: 0.2492 | MSE Test: 0.4930
Głębokość:   10 | MSE Train: 0.0555 | MSE Test: 0.3896
Głębokość:   20 | MSE Train: 0.0385 | MSE Test: 0.3820
Głębokość: Brak | MSE Train: 0.0384 | MSE Test: 0.3780

Parametr 3: Minimalna liczba próbek w liściu (min_samples_leaf)
Próbki w liściu:  1 | MSE Train: 0.0384 | MSE Test: 0.3780
Próbki w liściu:  2 | MSE Train: 0.0552 | MSE Test: 0.3879
Próbki w liściu:  5 | MSE Train: 0.1281 | MSE Test: 0.4246
Próbki w liściu: 10 | MSE Train: 0.2127 | MSE Test: 0.4759


METODA B: K-Najbliższych Sąsiadów (KNN Regressor)
Parametr 1: Liczba sąsiadów (n

In [29]:
print("=======================================================")
print("  CZĘŚĆ 2: PROBLEM 2 (KLASYFIKACJA - Klasa Jakości) ")
print("=======================================================\n")

print("METODA A: Las Losowy (Random Forest Classifier)")
print("Parametr 1: Liczba drzew (n_estimators)")
for n in [10, 50, 100, 200]:
    rfc = RandomForestClassifier(n_estimators=n, random_state=42)
    rfc.fit(X_train_c, y_train_c)
    print(f"Drzewa: {n:3d} | Skuteczność Train: {accuracy_score(y_train_c, rfc.predict(X_train_c))*100:.2f}% | Test: {accuracy_score(y_test_c, rfc.predict(X_test_c))*100:.2f}%")

print("\nParametr 2: Maksymalna głębokość drzewa (max_depth)")
for d in [5, 10, 20, None]:
    rfc = RandomForestClassifier(max_depth=d, random_state=42)
    rfc.fit(X_train_c, y_train_c)
    d_str = str(d) if d is not None else "Brak"
    print(f"Głębokość: {d_str:>4s} | Skuteczność Train: {accuracy_score(y_train_c, rfc.predict(X_train_c))*100:.2f}% | Test: {accuracy_score(y_test_c, rfc.predict(X_test_c))*100:.2f}%")

print("\nParametr 3: Minimalna liczba próbek w liściu (min_samples_leaf)")
for m in [1, 2, 5, 10]:
    rfc = RandomForestClassifier(min_samples_leaf=m, random_state=42)
    rfc.fit(X_train_c, y_train_c)
    print(f"Próbki w liściu: {m:2d} | Skuteczność Train: {accuracy_score(y_train_c, rfc.predict(X_train_c))*100:.2f}% | Test: {accuracy_score(y_test_c, rfc.predict(X_test_c))*100:.2f}%")


print("\n\nMETODA B: K-Najbliższych Sąsiadów (KNN Classifier)")
print("Parametr 1: Liczba sąsiadów (n_neighbors)")
for k in [3, 5, 9, 15]:
    knnc = KNeighborsClassifier(n_neighbors=k)
    knnc.fit(X_train_c, y_train_c)
    print(f"Sąsiedzi: {k:2d} | Skuteczność Train: {accuracy_score(y_train_c, knnc.predict(X_train_c))*100:.2f}% | Test: {accuracy_score(y_test_c, knnc.predict(X_test_c))*100:.2f}%")

print("\nParametr 2: Metryka odległości (p)")
for p_val in [1, 2, 3, 4]:
    knnc = KNeighborsClassifier(p=p_val)
    knnc.fit(X_train_c, y_train_c)
    print(f"Metryka p: {p_val:2d} | Skuteczność Train: {accuracy_score(y_train_c, knnc.predict(X_train_c))*100:.2f}% | Test: {accuracy_score(y_test_c, knnc.predict(X_test_c))*100:.2f}%")

print("\nParametr 3: Algorytm wyszukiwania sąsiadów")
for alg in ['ball_tree', 'kd_tree', 'brute', 'auto']:
    knnc = KNeighborsClassifier(algorithm=alg)
    knnc.fit(X_train_c, y_train_c)
    print(f"Algorytm: {alg:9s} | Skuteczność Train: {accuracy_score(y_train_c, knnc.predict(X_train_c))*100:.2f}% | Test: {accuracy_score(y_test_c, knnc.predict(X_test_c))*100:.2f}%")

  CZĘŚĆ 2: PROBLEM 2 (KLASYFIKACJA - Klasa Jakości) 

METODA A: Las Losowy (Random Forest Classifier)
Parametr 1: Liczba drzew (n_estimators)
Drzewa:  10 | Skuteczność Train: 99.23% | Test: 85.15%
Drzewa:  50 | Skuteczność Train: 100.00% | Test: 87.34%
Drzewa: 100 | Skuteczność Train: 100.00% | Test: 89.52%
Drzewa: 200 | Skuteczność Train: 100.00% | Test: 89.52%

Parametr 2: Maksymalna głębokość drzewa (max_depth)
Głębokość:    5 | Skuteczność Train: 89.17% | Test: 86.46%
Głębokość:   10 | Skuteczność Train: 97.70% | Test: 89.08%
Głębokość:   20 | Skuteczność Train: 100.00% | Test: 89.52%
Głębokość: Brak | Skuteczność Train: 100.00% | Test: 89.52%

Parametr 3: Minimalna liczba próbek w liściu (min_samples_leaf)
Próbki w liściu:  1 | Skuteczność Train: 100.00% | Test: 89.52%
Próbki w liściu:  2 | Skuteczność Train: 96.50% | Test: 89.08%
Próbki w liściu:  5 | Skuteczność Train: 91.03% | Test: 86.46%
Próbki w liściu: 10 | Skuteczność Train: 88.51% | Test: 86.90%


METODA B: K-Najbliższych

## Wnioski Końcowe
1. **Zjawisko Overfittingu (Przeuczenia):** Zaobserwowano ewidentne zjawisko przeuczenia. W autorskiej sieci SSN (dla regresji) błąd testowy zaczął rosnąć po przekroczeniu progu 32 neuronów. Znacznie silniejszy objaw wystąpił w metodzie Lasu Losowego – przy braku limitu głębokości drzewa (lub po osiągnięciu wartości 20), algorytm uczył się danych "na pamięć", uzyskując błąd MSE = 0.038 i 100% skuteczności na zbiorze treningowym. Model nadal potrafił generalizować wynik na próbę testową.
2. **Problem niezbalansowania klas w SSN:** Podczas testowania sieci neuronowej dla klasyfikacji zaobserwowano stagnację wyników na poziomie ok. 82-84% (Train) / 85-88% (Test). Wynika to z silnego niezbalansowania klas w zbiorze "Wine Quality", gdzie dominująca klasa (wina "średnie") stanowi przytłaczającą większość. Model znalazł lokalne optimum polegające na ciągłym zgadywaniu dominującej klasy.
3. **Parametry KNN a optymalizacja komputera:** Udowodniono empirycznie, że zmiana algorytmu wewnętrznego w KNN (z `auto` na `ball_tree` czy `brute`) nie wpływa na ostateczną skuteczność modelu (wynik pozostawał bez zmian). Oznacza to, że parametry te optymalizują jedynie indeksowanie pamięci i czas obliczeń (np. strukturę k-wymiarowego drzewa), a nie modyfikują przestrzeni wektorowej.
4. **Triumf uczenia zespołowego:** W testowanym zbiorze danych algorytm Random Forest okazał się bezkonkurencyjny. W problemie klasyfikacji z łatwością pobił skuteczność napisanego przez zespół Perceptronu (~90% przy 100 drzewach w stosunku do bariery 85% dla SSN).